# Notebook 01 — Data Exploration

Explore the basketball shot dataset after pose extraction and labeling.

**What we cover:**
- Shot outcome class balance (make / miss)
- Pose keypoint visibility statistics
- Video/clip statistics (frame count, fps distribution)
- Sample pose skeleton visualizations
- Shot type distribution

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
print('✅ Libraries loaded')

## 1. Load Labels & Manifests

In [ ]:
labels_path = Path('../data/annotations/shot_labels.csv')
labels = pd.read_csv(labels_path)

print(f'Total labeled shots: {len(labels)}')
print(f'Shot types: {labels["shot_type"].value_counts().to_dict()}')
print(f'\nMakes  : {labels["outcome"].sum()}')
print(f'Misses : {(1 - labels["outcome"]).sum()}')
print(f'Make % : {100 * labels["outcome"].mean():.1f}%')
print(f'\nLabel sources: {labels["label_source"].value_counts().to_dict()}')

## 2. Outcome Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = labels['outcome'].value_counts()
axes[0].pie(
    [counts.get(1, 0), counts.get(0, 0)],
    labels=['Make', 'Miss'],
    colors=['#2ecc71', '#e74c3c'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Overall Outcome Distribution')

# By shot type
by_type = labels.groupby(['shot_type', 'outcome']).size().unstack(fill_value=0)
by_type.columns = ['Miss', 'Make']
by_type.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[1].set_title('Outcomes by Shot Type')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Shot Dataset Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Pose Keypoint Visibility Analysis

MediaPipe assigns a visibility score [0-1] per landmark. Low visibility = unreliable coordinate.

In [ ]:
from pathlib import Path
import glob

# Load one sample pose CSV
pose_csvs = list(Path('../data/raw/poses').rglob('*_poses.csv'))
if pose_csvs:
    sample = pd.read_csv(pose_csvs[0])
    vis_cols = [c for c in sample.columns if c.endswith('_vis')]
    vis_means = sample[vis_cols].mean().sort_values()
    
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = ['#e74c3c' if v < 0.5 else '#f39c12' if v < 0.8 else '#2ecc71' 
              for v in vis_means.values]
    ax.barh(
        [c.replace('_vis', '') for c in vis_means.index],
        vis_means.values,
        color=colors, edgecolor='white'
    )
    ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Threshold=0.5')
    ax.axvline(0.8, color='green', linestyle='--', alpha=0.5, label='Target=0.8')
    ax.set_title(f'MediaPipe Landmark Visibility — {pose_csvs[0].name}', fontweight='bold')
    ax.set_xlabel('Mean Visibility Score [0-1]')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    low_vis = vis_means[vis_means < 0.5]
    print(f'Low visibility landmarks (< 0.5): {len(low_vis)}')
    if len(low_vis) > 0:
        print(low_vis.to_string())
else:
    print('No pose CSVs found. Run extract_poses.py first.')

## 4. Sequence Length Distribution

How long are the shot clips? This affects padding strategy for LSTM.

In [ ]:
norm_csvs = list(Path('../data/processed/keypoints').rglob('*_norm.csv'))
if norm_csvs:
    lengths = []
    for csv_path in norm_csvs:
        df = pd.read_csv(csv_path)
        lengths.append(len(df))
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(lengths, bins=30, color='#4C72B0', edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(lengths), color='red', linestyle='--', 
               label=f'Mean: {np.mean(lengths):.0f} frames')
    ax.axvline(30, color='green', linestyle='--', alpha=0.8,
               label='LSTM window: 30 frames')
    ax.set_title('Shot Clip Length Distribution', fontweight='bold')
    ax.set_xlabel('Clip length (frames)')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print(f'Total clips   : {len(lengths)}')
    print(f'Mean length   : {np.mean(lengths):.1f} frames ({np.mean(lengths)/30:.1f}s at 30fps)')
    print(f'Median length : {np.median(lengths):.1f} frames')
    print(f'Min / Max     : {min(lengths)} / {max(lengths)} frames')
    print(f'% clips < 30f : {100*sum(l < 30 for l in lengths)/len(lengths):.1f}% (need padding)')
else:
    print('No normalized CSVs found. Run normalize_poses.py first.')

## 5. Sample Pose Visualization

Draw MediaPipe skeleton on a sample frame to verify pose quality.

In [ ]:
import cv2

# MediaPipe connections for drawing skeleton
CONNECTIONS = [
    ('right_shoulder', 'right_elbow'), ('right_elbow', 'right_wrist'),
    ('left_shoulder', 'left_elbow'), ('left_elbow', 'left_wrist'),
    ('right_shoulder', 'left_shoulder'),
    ('right_hip', 'left_hip'),
    ('right_shoulder', 'right_hip'), ('left_shoulder', 'left_hip'),
    ('right_hip', 'right_knee'), ('right_knee', 'right_ankle'),
    ('left_hip', 'left_knee'), ('left_knee', 'left_ankle'),
]

def draw_skeleton(ax, row, img_w=640, img_h=480, color='#2ecc71'):
    """Draw pose skeleton from a normalized pose row."""
    for a, b in CONNECTIONS:
        x1 = row.get(f'{a}_x', np.nan)
        y1 = row.get(f'{a}_y', np.nan)
        x2 = row.get(f'{b}_x', np.nan)
        y2 = row.get(f'{b}_y', np.nan)
        if not any(np.isnan([x1, y1, x2, y2])):
            ax.plot([x1*img_w, x2*img_w], [y1*img_h, y2*img_h],
                    '-', color=color, linewidth=2, alpha=0.8)
    for lm in ['right_shoulder','right_elbow','right_wrist',
               'right_hip','right_knee','right_ankle']:
        x = row.get(f'{lm}_x', np.nan)
        y = row.get(f'{lm}_y', np.nan)
        if not np.isnan(x) and not np.isnan(y):
            ax.plot(x*img_w, y*img_h, 'o', color='white', markersize=6, 
                    markeredgecolor=color, markeredgewidth=2)

if norm_csvs:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    import random
    selected = random.sample(norm_csvs, min(8, len(norm_csvs)))
    
    for ax, csv_path in zip(axes.flatten(), selected):
        df = pd.read_csv(csv_path)
        if len(df) > 0:
            mid_row = df.iloc[len(df)//2]
            ax.set_facecolor('#1a1a2e')
            ax.set_xlim(0, 640); ax.set_ylim(480, 0)
            draw_skeleton(ax, mid_row)
            ax.set_title(csv_path.stem[:25], fontsize=8)
            ax.axis('off')
    
    plt.suptitle('Sample Normalized Pose Skeletons (mid-shot frame)', 
                 fontsize=13, fontweight='bold', color='white')
    fig.patch.set_facecolor('#0d0d1a')
    plt.tight_layout()
    plt.show()
else:
    print('No pose data to visualize yet.')